In [ ]:
import pandas as pd
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import time
import re
import warnings
warnings.filterwarnings('ignore')

performance = pd.DataFrame(columns=['Name', 'Accuracy', 'Precision', 'Sensitivity', 'F1 Score', 'MCC', 'Markedness', "Youden's J", 'FMI', 'Time'])

df = pd.read_csv("StealthPhisher2025.csv")

LABEL = df.iloc[:,-1:].columns[0]

fs = ['LengthOfURL', 'URLComplexity', 'CharacterComplexity','DomainLengthOfURL', 'IsDomainIP', 'TLDLength', 'LetterCntInURL',
       'URLLetterRatio', 'DigitCntInURL', 'URLDigitRatio', 'EqualCharCntInURL', 'QuesMarkCntInURL', 'AmpCharCntInURL', 'OtherSpclCharCntInURL',
       'URLOtherSpclCharRatio', 'NumberOfHashtags', 'NumberOfSubdomains', 'HavingPath', 'PathLength', 'HavingQuery', 'HavingFragment',
       'HavingAnchor', 'HasSSL', 'IsUnreachable', 'LineOfCode', 'LongestLineLength', 'HasTitle', 'HasFavicon',LABEL]

df = pd.DataFrame(df[fs]).copy()
print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, roc_curve
)
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import time
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

start_time = time.time()

performance = pd.DataFrame(columns=['Name', 'Accuracy', 'Precision', 'Sensitivity', 'F1 Score', 'MCC', 'Markedness', "Youden's J", 'FMI', 'Time'])

X = df.drop(columns=[LABEL]).values
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def MarkednessIndex(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def YoudenJStatistic(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def FowlkesMallowsIndex(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
def BuildWideDeepModel(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = BuildWideDeepModel(X_train_scaled.shape[1])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  MarkednessIndex(y_test, y_pred)
youden_j = YoudenJStatistic(y_test, y_pred)
fmi = FowlkesMallowsIndex(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

adv_start_time = time.time()
features = df.drop(columns=["Label"]).values
labels = df["Label"].values

features_tensor = torch.tensor(features, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

class SimpleTabularNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleTabularNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# advModel parameters
input_size = features.shape[1]  # Number of features in the dataset
hidden_size = 64  # Example hidden layer size
output_size = len(set(labels))  # Number of classes (if it's a classification problem)

# Initialize the advModel, loss function, and optimizer
advModel = SimpleTabularNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(advModel.parameters(), lr=0.001)

# Define FGSM attack function
def fgsm_attack(advModel, data, labels, epsilon=0.1):
    # Forward pass through the advModel
    outputs = advModel(data)
    advModel.zero_grad()

    # Calculate the loss
    loss = criterion(outputs, labels)
    loss.backward()

    # Collect the sign of the gradients
    sign_data_grad = data.grad.data.sign()

    # Create the perturbed data by adding the gradient sign
    perturbed_data = data + epsilon * sign_data_grad
    perturbed_data = torch.clamp(perturbed_data, min=0, max=1)  # Keep features within [0, 1] range if needed

    return perturbed_data

# Generate adversarial examples using FGSM
adversarial_data = []
adversarial_labels = []

advModel.eval()  # Set advModel to evaluation mode for inference

# Loop through the dataset in batches (batch size 32)
for i in range(0, len(features_tensor), 32):  # Batch size of 32
    batch_data = features_tensor[i:i+32]
    batch_labels = labels_tensor[i:i+32]

    # Create a new tensor with requires_grad=True for adversarial attack
    batch_data = batch_data.clone().detach().requires_grad_(True)

    # Generate adversarial examples
    perturbed_batch = fgsm_attack(advModel, batch_data, batch_labels, epsilon=0.1)

    # Store adversarial data and the corresponding labels
    adversarial_data.append(perturbed_batch)
    adversarial_labels.append(batch_labels)

# Concatenate all the adversarial examples
adversarial_data = torch.cat(adversarial_data)
adversarial_labels = torch.cat(adversarial_labels)

adversarial_df = pd.DataFrame(adversarial_data.detach().numpy(), columns=df.drop(columns=["Label"]).columns)
adversarial_labels_df = pd.DataFrame(adversarial_labels.numpy(), columns=["Label"])

adv_df = pd.concat([adversarial_df, adversarial_labels_df], axis=1)
adv_df.to_csv('StealthPhisher2025_FGSM_Adversarial_Attack_Dataset.csv',index=False)

print("Adversarial dataset generated!")

adversarial_df = scaler.fit_transform(adversarial_df)

adv_pred = (model.predict([adversarial_df, adversarial_df]) > 0.5).astype("int32")

accuracy = accuracy_score(adversarial_labels_df, adv_pred)
precision = precision_score(adversarial_labels_df, adv_pred)
sensitivity = recall_score(adversarial_labels_df, adv_pred)
f1 = f1_score(adversarial_labels_df, adv_pred)
mcc = matthews_corrcoef(adversarial_labels_df, adv_pred)
roc_auc = roc_auc_score(adversarial_labels_df, adv_pred)
conf_matrix = confusion_matrix(adversarial_labels_df, adv_pred)
markedness =  MarkednessIndex(adversarial_labels_df, adv_pred)
youden_j = YoudenJStatistic(adversarial_labels_df, adv_pred)
fmi = FowlkesMallowsIndex(adversarial_labels_df, adv_pred)

adv_end_time = time.time()
adv_elapsed_time = adv_end_time - adv_start_time

new_row = pd.DataFrame([{
    'Name': 'Robustness against FGSM Adversarial Attack Dataset',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': adv_elapsed_time
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance